Vamos a usar un dataset sintético para la uc4, tiene un patrón scatter-gather. Las cuentas son bank-account y van de la 1-11 a la 8-18. La cuenta origen del patrón es la 1-11 y la de destino es la 1-18, o sea, 1-11 manda a las cuentas 2-12 a 7-17 y esas cuentas le mandan a la 8-18.

In [7]:
import pandas as pd

transactions_df = pd.read_csv('uc4_sintetico.csv')
transactions_df

,Unnamed: 0,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,0,2022/09/02 00:08,1,11,2,12,100.0,US Dollar,100.0,US Dollar,Reinvestment,0
1,0,2022/09/02 00:08,1,11,3,13,100.0,US Dollar,100.0,US Dollar,Reinvestment,0
2,0,2022/09/02 00:08,1,11,4,14,100.0,US Dollar,100.0,US Dollar,Reinvestment,0
3,0,2022/09/02 00:08,1,11,5,15,100.0,US Dollar,100.0,US Dollar,Reinvestment,0
4,0,2022/09/02 00:08,1,11,6,16,100.0,US Dollar,100.0,US Dollar,Reinvestment,0
5,0,2022/09/02 00:08,1,11,7,17,100.0,US Dollar,100.0,US Dollar,Reinvestment,0
6,0,2022/09/02 00:08,2,12,8,18,100.0,US Dollar,100.0,US Dollar,Reinvestment,0
7,0,2022/09/02 00:08,3,13,8,18,100.0,US Dollar,100.0,US Dollar,Reinvestment,0
8,0,2022/09/02 00:08,4,14,8,18,100.0,US Dollar,100.0,US Dollar,Reinvestment,0
9,0,2022/09/02 00:08,5,15,8,18,100.0,US Dollar,100.0,US Dollar,Reinvestment,0


Primero mostramos que el procesamiento que se hacía en el notebook original no agarra esta ocurrencia del patrón.

In [8]:
usd_df = transactions_df[transactions_df["Payment Currency"] == "US Dollar"]
usd_period_a_df = usd_df[
    (usd_df["Timestamp"] >= "2022/09/01")
    & (usd_df["Timestamp"] <= "2022/09/06")
]

target_transactions_df = usd_period_a_df

target_transactions_df # van todas

,Unnamed: 0,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,0,2022/09/02 00:08,1,11,2,12,100.0,US Dollar,100.0,US Dollar,Reinvestment,0
1,0,2022/09/02 00:08,1,11,3,13,100.0,US Dollar,100.0,US Dollar,Reinvestment,0
2,0,2022/09/02 00:08,1,11,4,14,100.0,US Dollar,100.0,US Dollar,Reinvestment,0
3,0,2022/09/02 00:08,1,11,5,15,100.0,US Dollar,100.0,US Dollar,Reinvestment,0
4,0,2022/09/02 00:08,1,11,6,16,100.0,US Dollar,100.0,US Dollar,Reinvestment,0
5,0,2022/09/02 00:08,1,11,7,17,100.0,US Dollar,100.0,US Dollar,Reinvestment,0
6,0,2022/09/02 00:08,2,12,8,18,100.0,US Dollar,100.0,US Dollar,Reinvestment,0
7,0,2022/09/02 00:08,3,13,8,18,100.0,US Dollar,100.0,US Dollar,Reinvestment,0
8,0,2022/09/02 00:08,4,14,8,18,100.0,US Dollar,100.0,US Dollar,Reinvestment,0
9,0,2022/09/02 00:08,5,15,8,18,100.0,US Dollar,100.0,US Dollar,Reinvestment,0


Acá viene el problema. Ahora se queda con las transacciones para las cuales la cuenta origen mandó a entre 5 y 10 cuentas distintas.
El lower bound 5 es exclusivo y el upper bound 10 directamente no está en el enunciado, sin embargo, el dataset a propósito tiene un scatter-gather de 6 paths.

**Nos quedamos con el scatter pero se descartó por completo el gather**.

In [9]:
def filter_function(x):
    unique_account_size = x.groupby(["To Bank", "Account.1"]).size().size
    return unique_account_size > 5 and unique_account_size < 10

target_transactions_df = target_transactions_df.groupby(
    ["From Bank", "Account"]
).filter(filter_function)

target_transactions_df = target_transactions_df[
    ["From Bank", "Account", "To Bank", "Account.1"]
]

target_transactions_df

,From Bank,Account,To Bank,Account.1
0,1,11,2,12
1,1,11,3,13
2,1,11,4,14
3,1,11,5,15
4,1,11,6,16
5,1,11,7,17


In [10]:
# acá los one length paths sobre el mismo dataset, mergea transacciones que
# mandaron a una cuenta con transacciones que recibieron de esa cuenta, ya 
# no hay ningún path porque ya descartamos el gather
paths_df = target_transactions_df.merge(
    target_transactions_df, left_on=["To Bank", "Account.1"], right_on=["From Bank", "Account"]
).rename(
    columns={
        "From Bank_x": "From Bank",
        "Account_x": "From Account",
        "To Bank_y": "To Bank",
        "Account.1_y": "To Account",
    }
)

# descarta las auto transacciones (no hay)
paths_df = paths_df[
    (paths_df["From Bank"] != paths_df["To Bank"])
    | (paths_df["From Account"] != paths_df["To Account"])
]

paths_df # ya no hay nada

,From Bank,From Account,To Bank_x,Account.1_x,From Bank_y,Account_y,To Bank,To Account


Sólo por completitud, sigue el resto del análisis

In [11]:
# para cada fila, o sea para cada path, cuenta las ocurrencias
paths_df = paths_df.groupby(
    ["From Bank", "From Account", "To Bank", "To Account"], as_index=False
).size()

# se queda con los paths que tengan más de 5 ocurrencias
paths_df = paths_df[(paths_df["size"] > 5)]

# el resultado es la cuenta origen y la cuenta destino de cada path
from_account_pairs_df = paths_df[["From Bank", "From Account"]].rename(
    columns={"From Bank": "Bank", "From Account": "Account"}
)
to_account_pairs_df = paths_df[["To Bank", "To Account"]].rename(
    columns={"To Bank": "Bank", "To Account": "Account"}
)

result = pd.concat([from_account_pairs_df, to_account_pairs_df]).drop_duplicates()

result

,Bank,Account


La solución es simplemente eliminar el filtro que borra el gather:

In [12]:
usd_df = transactions_df[transactions_df["Payment Currency"] == "US Dollar"]
usd_period_a_df = usd_df[
    (usd_df["Timestamp"] >= "2022/09/01")
    & (usd_df["Timestamp"] <= "2022/09/06")
]

target_transactions_df = usd_period_a_df

# def filter_function(x):
#     unique_account_size = x.groupby(["To Bank", "Account.1"]).size().size
#     return unique_account_size > 5 and unique_account_size < 10

# target_transactions_df = target_transactions_df.groupby(
#     ["From Bank", "Account"]
# ).filter(filter_function)

target_transactions_df = target_transactions_df[
    ["From Bank", "Account", "To Bank", "Account.1"]
]

paths_df = target_transactions_df.merge(
    target_transactions_df, left_on=["To Bank", "Account.1"],
    right_on=["From Bank", "Account"]
).rename(
    columns={
        "From Bank_x": "From Bank",
        "Account_x": "From Account",
        "To Bank_y": "To Bank",
        "Account.1_y": "To Account",
    }
)

paths_df = paths_df[
    (paths_df["From Bank"] != paths_df["To Bank"])
    | (paths_df["From Account"] != paths_df["To Account"])
]

paths_df = paths_df.groupby(
    ["From Bank", "From Account", "To Bank", "To Account"], as_index=False
).size()

paths_df = paths_df[(paths_df["size"] > 5)]

from_account_pairs_df = paths_df[["From Bank", "From Account"]].rename(
    columns={"From Bank": "Bank", "From Account": "Account"}
)
to_account_pairs_df = paths_df[["To Bank", "To Account"]].rename(
    columns={"To Bank": "Bank", "To Account": "Account"}
)

result = pd.concat([from_account_pairs_df, to_account_pairs_df]).drop_duplicates()

result

,Bank,Account
0,1,11
0,8,18
